# Trustworthy AI — Weeks 8–9
## Data Attribution and Model Attribution

**A hands-on companion notebook to the lecture.**

Attribution answers *"why did the model do that?"* in two ways:

- **Data attribution** — how the *training data* shaped the model (which samples mattered).
- **Model attribution** — which *factors inside the model* (features, gradients, attention)
  drove a specific prediction.

Each section states the slide idea, then demonstrates it with a *tiny, self-contained* model
so everything runs in a few seconds on a free Colab GPU.

> **How to use:** `Runtime → Run all`, or run section by section. The **Setup** cells must run
> first; after them the two parts are independent.

---

### Roadmap (mirrors the lecture)

**Part 1 — Data Attribution**
1. Influence Functions
2. Adversarial Examples (fooling the model — text)
3. Counterfactual Explanations
4. Data Shapley Values
5. Prototype & Criticism Selection

**Part 2 — Model Attribution**
6. Feature Importance (Permutation, LIME, SHAP)
7. Gradient-based Attribution (Saliency, Integrated Gradients, Grad-CAM)
8. Surrogate Models
9. Attention Mechanisms

---
*Companion to `W8_9_DataModelAttribution`. Educational; all models are toy models we train
ourselves so everything is fast and reproducible.*

## 0. Setup

Enable the GPU: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**.
The image demos place tensors on `DEVICE` (the GPU when available).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU for the image sections.")

---
# Part 1 — Data Attribution

> *"How the training data influences the model's predictions and behavior."*

Data attribution traces a prediction back to the **training points** responsible for it —
essential for debugging mislabeled or biased data, valuing data, and cleaning datasets.

## 1. Influence Functions

**Slide idea:** measure how each *individual training point* affects model predictions /
test loss, using second-order derivatives (the Hessian). Useful for debugging mislabeled or
biased data. *(Koh & Liang, 2017 — arxiv 1703.04730.)*

Influence of training point *z* on a test point *z_test* is
`I(z, z_test) = −∇θ L(z_test)ᵀ H⁻¹ ∇θ L(z)`.
We compute it exactly on a small **logistic-regression** model where the Hessian is tiny, then
show it flags a deliberately **mislabeled** training point.

In [ ]:
from sklearn.datasets import make_classification

# Small 2-class problem so the Hessian is a small, invertible matrix.
Xd, yd = make_classification(n_samples=200, n_features=5, n_informative=4,
                             n_redundant=0, random_state=0)
Xd = torch.tensor(Xd, dtype=torch.float32); yd = torch.tensor(yd, dtype=torch.float32)

# Poison one training point: flip its label (this is the "mislabeled" sample to catch).
POISON = 7
yd_bad = yd.clone(); yd_bad[POISON] = 1 - yd_bad[POISON]

w = torch.zeros(5, requires_grad=True); b = torch.zeros(1, requires_grad=True)
def logits(X): return X @ w + b
def loss_fn(X, y): return F.binary_cross_entropy_with_logits(logits(X), y)

opt = torch.optim.LBFGS([w, b], lr=0.5, max_iter=100)
def closure():
    opt.zero_grad(); l = loss_fn(Xd, yd_bad); l.backward(); return l
opt.step(closure)
print("Trained logistic regression. Final loss:", loss_fn(Xd, yd_bad).item())

# Exact influence via the full Hessian of the training loss w.r.t. params.
params = torch.cat([w, b])
def total_loss(p):
    ww, bb = p[:5], p[5:]
    return F.binary_cross_entropy_with_logits(Xd @ ww + bb, yd_bad)

from torch.autograd.functional import hessian
H = hessian(total_loss, params.detach()) + 1e-3*torch.eye(6)   # damping for invertibility
H_inv = torch.inverse(H)

def grad_params(X, y):
    p = params.detach().clone().requires_grad_(True)
    l = F.binary_cross_entropy_with_logits(X @ p[:5] + p[5:], y)
    return torch.autograd.grad(l, p)[0]

# Influence of every training point on the mean test loss of a held-out set.
Xtest, ytest = Xd[150:], yd[150:]
g_test = grad_params(Xtest, ytest)
influences = []
for i in range(150):
    g_i = grad_params(Xd[i:i+1], yd_bad[i:i+1])
    influences.append(-(g_test @ H_inv @ g_i).item())
influences = np.array(influences)

rank = influences.argsort()          # most harmful (negative influence) first...
print(f"\nMost HARMFUL training points (raise test loss): {influences.argsort()[:5].tolist()}")
print(f"Poisoned/mislabeled point index was: {POISON}")
print(f"Its influence rank (0=most harmful): {list(influences.argsort()).index(POISON)}  out of 150")
plt.figure(figsize=(7,2.5)); plt.bar(range(150), influences)
plt.axvline(POISON, color="red", ls="--", label=f"mislabeled point #{POISON}")
plt.xlabel("training point"); plt.ylabel("influence on test loss"); plt.legend()
plt.title("Influence functions flag the mislabeled sample"); plt.show()

## 2. Adversarial Examples — Fooling the Model (Text)

**Slide idea:** subtly modify the input to test robustness and find weaknesses. **TextFooler**
replaces important words with synonyms until the prediction flips — e.g. *contrived→engineered*,
*totally→fully* turned a Negative review Positive.

We build a tiny sentiment classifier and a mini TextFooler: rank words by how much removing
each drops the confidence, then swap the top word for a synonym.

In [ ]:
# Tiny bag-of-words sentiment model trained on a toy labelled set.
pos = ["great movie", "i love this", "wonderful and fun", "an excellent film",
       "brilliant acting superb", "amazing truly enjoyable", "fantastic wonderful story"]
neg = ["terrible movie", "i hate this", "boring and dull", "an awful film",
       "poor acting weak", "dreadful truly painful", "horrible boring story"]
texts = pos + neg; labels = [1]*len(pos) + [0]*len(neg)

vocab = sorted(set(w for t in texts for w in t.split()))
w2i = {w:i for i,w in enumerate(vocab)}
def vec(t):
    v = torch.zeros(len(vocab))
    for w in t.split():
        if w in w2i: v[w2i[w]] += 1
    return v
Xtxt = torch.stack([vec(t) for t in texts]); ytxt = torch.tensor(labels, dtype=torch.float32)

clf = nn.Linear(len(vocab), 1)
opt = torch.optim.Adam(clf.parameters(), 0.1)
for _ in range(300):
    opt.zero_grad(); F.binary_cross_entropy_with_logits(clf(Xtxt).squeeze(), ytxt).backward(); opt.step()

def predict(t):
    p = torch.sigmoid(clf(vec(t))).item()
    return ("positive" if p>0.5 else "negative"), p

# A tiny synonym table (real TextFooler uses word embeddings + POS constraints).
SYN = {"great":"decent", "love":"tolerate", "wonderful":"okay", "excellent":"passable",
       "brilliant":"adequate", "amazing":"fine", "fantastic":"acceptable"}

def textfooler(t):
    base_lbl, base_p = predict(t)
    words = t.split()
    # importance = drop in confidence when a word is removed
    imp = []
    for i,wd in enumerate(words):
        without = " ".join(words[:i]+words[i+1:])
        imp.append(base_p - torch.sigmoid(clf(vec(without))).item())
    order = np.argsort(imp)[::-1]         # most important first
    new = words[:]
    for i in order:
        if new[i] in SYN:
            new[i] = SYN[new[i]]
            cand = " ".join(new)
            if predict(cand)[0] != base_lbl:
                return t, cand, base_lbl, predict(cand)[0]
    return t, " ".join(new), base_lbl, predict(" ".join(new))[0]

orig, adv, l0, l1 = textfooler("great wonderful story")
print(f"Original : '{orig}'  -> {l0}  ({predict(orig)[1]:.2f})")
print(f"Perturbed: '{adv}'  -> {l1}  ({predict(adv)[1]:.2f})")
print("Swapping high-importance words for weaker synonyms flips the sentiment prediction.")

## 3. Counterfactual Explanations

**Slide idea:** find the *minimal change* to the input that flips the outcome —
*"if income were \$5,000 higher, the loan would be approved."* Two steps: identify important
tokens, then modify them. (Polyjuice adds/deletes/negates words.)

We do a tabular counterfactual on a loan model: search the smallest feature change that flips
a rejection to an approval.

In [ ]:
# Toy loan model: approve if a linear score passes a threshold.
rng = np.random.RandomState(0)
feat_names = ["income", "debt", "credit_score", "years_employed"]
Xl = rng.rand(400, 4); Xl[:,2] *= 850; Xl[:,0]*=100000; Xl[:,1]*=50000; Xl[:,3]*=20
true_w = np.array([2e-5, -3e-5, 0.004, 0.05])
yl = ((Xl @ true_w - 1.0) > 0).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(Xl)
loan = LogisticRegression().fit(sc.transform(Xl), yl)

def approve(x): return loan.predict(sc.transform([x]))[0], loan.predict_proba(sc.transform([x]))[0,1]

# Pick a rejected applicant and find the minimal single-feature change to get approved.
rejected = Xl[yl==0][3].copy()
print("Applicant:", {n:round(v,1) for n,v in zip(feat_names, rejected)})
print("Decision :", "APPROVED" if approve(rejected)[0] else "REJECTED",
      f"(p={approve(rejected)[1]:.2f})\n")

best = None
for j, name in enumerate(feat_names):
    x = rejected.copy(); step = rejected[j]*0.02 + 1
    for _ in range(500):
        x[j] += step
        if approve(x)[0] == 1:
            delta = x[j]-rejected[j]
            if best is None or abs(delta) < best[2]:
                best = (name, x[j], abs(delta))
            break
print(f"Counterfactual: increase '{best[0]}' to {best[1]:.0f} "
      f"(change of {best[2]:.0f}) -> loan APPROVED")
print("This is the minimal single-feature nudge that flips the decision.")

## 4. Data Shapley Values

**Slide idea:** assign a *value* to each training point by its contribution to model
performance — helps data selection and cleaning. Grounded in the Shapley value from
cooperative game theory.

Exact Shapley is exponential, so we use the efficient **KNN-Shapley** closed form (Jia et al.,
2019). Good/clean points get positive value; mislabeled points get *negative* value.

In [ ]:
# Reuse the tabular data; inject a few mislabeled points to see negative Shapley values.
from sklearn.model_selection import train_test_split
Xs, ys = make_classification(n_samples=300, n_features=6, n_informative=5,
                             n_redundant=0, random_state=1)
Xtr_s, Xval_s, ytr_s, yval_s = train_test_split(Xs, ys, test_size=100, random_state=0)
ytr_s = ytr_s.copy()
mislabeled = [2, 10, 25, 40]
for i in mislabeled: ytr_s[i] = 1 - ytr_s[i]

def knn_shapley(Xtr, ytr, Xval, yval, K=5):
    # Closed-form KNN-Shapley: value of each training point averaged over validation queries.
    N = len(Xtr); vals = np.zeros(N)
    for xv, yvv in zip(Xval, yval):
        d = np.linalg.norm(Xtr - xv, axis=1)
        idx = np.argsort(d)                       # nearest first
        s = np.zeros(N)
        # recursion from farthest to nearest (Jia et al. 2019, Eq. for KNN-Shapley)
        s[idx[-1]] = (ytr[idx[-1]] == yvv) / N
        for j in range(N-2, -1, -1):
            i, i1 = idx[j], idx[j+1]
            s[i] = s[i1] + ((ytr[i]==yvv) - (ytr[i1]==yvv)) / K * min(K, j+1)/(j+1)
        vals += s
    return vals / len(Xval)

phi = knn_shapley(Xtr_s, ytr_s, Xval_s, yval_s)
print("Lowest-value (likely mislabeled) training points:", phi.argsort()[:6].tolist())
print("Injected mislabeled indices                     :", mislabeled)

plt.figure(figsize=(7,2.5)); plt.bar(range(len(phi)), phi)
for i in mislabeled: plt.axvline(i, color="red", ls="--", alpha=0.6)
plt.xlabel("training point"); plt.ylabel("Data Shapley value")
plt.title("Data Shapley: mislabeled points (red) get negative value"); plt.show()

## 5. Prototype & Criticism Selection

**Slide idea:** identify *prototypes* (representative samples that summarize a class) and
*criticisms* (atypical points the prototypes fail to explain). Helps understand class
boundaries. (MMD-critic, Kim et al., 2016.)

We select prototypes greedily to match the data distribution (max-MMD) and criticisms as the
points worst represented by those prototypes — on MNIST digit images.

In [ ]:
from torchvision import datasets, transforms
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
Xm = (mnist.data.float()/255.0).reshape(-1, 784)
ym = mnist.targets

# Work within one class (digit 3) so prototypes/criticisms are interpretable.
cls = 3
Xc = Xm[ym==cls][:500]

def rbf(A, B, gamma=0.5):
    d = torch.cdist(A, B)**2
    return torch.exp(-gamma*d)

def select_prototypes(X, m=8):
    # Greedy MMD: add the point that best increases similarity to the whole set.
    n = len(X); K = rbf(X, X); mean_k = K.mean(0)
    chosen = []
    for _ in range(m):
        best, best_score = None, -1e9
        for i in range(n):
            if i in chosen: continue
            S = chosen + [i]
            score = 2*mean_k[S].sum()/len(S) - K[S][:,S].sum()/len(S)**2
            if score > best_score: best_score, best = score, i
        chosen.append(best)
    return chosen

def select_criticisms(X, protos, m=8):
    # Points where the prototype density least matches the data density.
    K = rbf(X, X); Kp = rbf(X, X[protos])
    witness = K.mean(1) - Kp.mean(1)              # data density - prototype density
    return witness.abs().argsort(descending=True)[:m].tolist()

protos = select_prototypes(Xc, 8)
crits  = select_criticisms(Xc, protos, 8)

fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
for j,p in enumerate(protos):
    ax[0,j].imshow(Xc[p].reshape(28,28), cmap="gray"); ax[0,j].axis("off")
for j,c in enumerate(crits):
    ax[1,j].imshow(Xc[c].reshape(28,28), cmap="gray"); ax[1,j].axis("off")
ax[0,0].set_title("prototypes (typical 3s)", loc="left")
ax[1,0].set_title("criticisms (unusual 3s)", loc="left")
plt.suptitle("Prototype & criticism selection for digit '3'"); plt.tight_layout(); plt.show()

---
# Part 2 — Model Attribution

> *"The factors within the model that influence its decisions — weights, feature importance,
> gradients."*

Model attribution explains a *specific prediction* in terms of the model's internal signals:
which input features, which gradients, which attention weights drove the output.

### A shared CNN + MLP

Gradient methods need an image model; feature-importance methods use a tabular model. We train
both once here.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Image model (for saliency / IG / Grad-CAM).
Ximg = (mnist.data.float()/255.0).unsqueeze(1)
yimg = mnist.targets

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU())          # keep last conv for Grad-CAM (28->14)
        self.pool = nn.MaxPool2d(2)                            # 14->7
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7,64), nn.ReLU(), nn.Linear(64,10))
    def forward(self, x):
        f = self.features(x); return self.classifier(self.pool(f))

cnn = CNN().to(DEVICE)
opt = torch.optim.Adam(cnn.parameters(), 1e-3)
loader = DataLoader(TensorDataset(Ximg[:20000], yimg[:20000]), batch_size=256, shuffle=True)
print("Training CNN...")
for ep in range(2):
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        opt.zero_grad(); F.cross_entropy(cnn(xb),yb).backward(); opt.step()
acc = (cnn(Ximg[50000:51000].to(DEVICE)).argmax(1).cpu()==yimg[50000:51000]).float().mean()
print(f"CNN test-ish accuracy: {acc:.3f}")

## 6. Feature Importance (Permutation, LIME, SHAP)

**Slide idea:** measure how much each input feature contributes to the output — via **SHAP**,
**LIME**, and **permutation importance**. We compare all three on the tabular loan-style model,
where features have names.

In [ ]:
# A richer tabular model with named features (random forest so importances are interesting).
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
Xf, yf = make_classification(n_samples=800, n_features=6, n_informative=4, n_redundant=1,
                             random_state=3)
fnames = [f"feat_{i}" for i in range(6)]
rf = RandomForestClassifier(n_estimators=120, random_state=0).fit(Xf[:600], yf[:600])

# (a) Permutation importance
perm = permutation_importance(rf, Xf[600:], yf[600:], n_repeats=10, random_state=0)

# (b) LIME (local, one instance) -- fit a weighted linear model on perturbations around x0.
def lime_explain(model, x0, n=1000, sigma=0.5):
    Z = x0 + sigma*np.random.randn(n, len(x0))
    p = model.predict_proba(Z)[:,1]
    wts = np.exp(-np.linalg.norm(Z - x0, axis=1)**2 / (2*(sigma*2)**2))
    from sklearn.linear_model import Ridge
    lin = Ridge(alpha=1.0).fit(Z*wts[:,None], p*wts)
    return lin.coef_

x0 = Xf[600]
lime_w = lime_explain(rf, x0)

# (c) SHAP via a small KernelSHAP-style sampling (marginal contribution over random coalitions).
def kernel_shap(model, x0, background, n=400):
    d = len(x0); phi = np.zeros(d)
    base = model.predict_proba(background)[:,1].mean()
    for _ in range(n):
        perm_order = np.random.permutation(d)
        z = background[np.random.randint(len(background))].copy()
        prev = model.predict_proba([z])[0,1]
        for j in perm_order:
            z[j] = x0[j]
            cur = model.predict_proba([z])[0,1]
            phi[j] += cur - prev; prev = cur
    return phi/n

shap_w = kernel_shap(rf, x0, Xf[:600])

fig, ax = plt.subplots(1,3, figsize=(13,3))
ax[0].barh(fnames, perm.importances_mean); ax[0].set_title("Permutation importance (global)")
ax[1].barh(fnames, lime_w);  ax[1].set_title("LIME weights (local, this instance)")
ax[2].barh(fnames, shap_w);  ax[2].set_title("SHAP values (local, this instance)")
plt.tight_layout(); plt.show()
print("Permutation = global; LIME & SHAP = per-instance. They often agree on the top features.")

## 7. Gradient-based Attribution (Saliency, Integrated Gradients, Grad-CAM)

**Slide idea:** use gradients to see how input changes affect the prediction — **Saliency
maps**, **Integrated Gradients**, and **Grad-CAM** (for CNNs). We compute all three on the same
MNIST image and its predicted class.

In [ ]:
img = Ximg[50003:50004].to(DEVICE)     # a single test image
target = cnn(img).argmax(1).item()

# (a) Saliency: |gradient of the class score w.r.t. the input|.
x = img.clone().requires_grad_(True)
cnn(x)[0, target].backward()
saliency = x.grad.abs()[0,0].cpu()

# (b) Integrated Gradients: integrate gradients along a path from a black baseline to the image.
def integrated_gradients(model, x, target, steps=50):
    baseline = torch.zeros_like(x)
    total = torch.zeros_like(x)
    for a in torch.linspace(0,1,steps):
        xi = (baseline + a*(x-baseline)).requires_grad_(True)
        model(xi)[0, target].backward()
        total += xi.grad
    return ((x-baseline)*total/steps)[0,0].detach().cpu().abs()
ig = integrated_gradients(cnn, img, target)

# (c) Grad-CAM: weight the last conv feature maps by their gradient, ReLU, upsample.
feat_store = {}
h = cnn.features.register_forward_hook(lambda m,i,o: feat_store.setdefault("f", o))
img2 = img.clone().requires_grad_(True)
out = cnn(img2); cnn.zero_grad(); out[0,target].backward()
fmap = feat_store["f"]                          # (1,32,14,14)
grads = torch.autograd.grad(out[0,target], fmap, retain_graph=True)[0]
weights = grads.mean(dim=(2,3), keepdim=True)
cam = F.relu((weights*fmap).sum(1))[0]
cam = F.interpolate(cam[None,None], size=(28,28), mode="bilinear")[0,0].detach().cpu()
cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
h.remove()

fig, ax = plt.subplots(1,4, figsize=(12,3))
ax[0].imshow(img[0,0].cpu(), cmap="gray"); ax[0].set_title(f"input (pred={target})")
ax[1].imshow(saliency, cmap="hot"); ax[1].set_title("Saliency")
ax[2].imshow(ig, cmap="hot"); ax[2].set_title("Integrated Gradients")
ax[3].imshow(img[0,0].cpu(), cmap="gray"); ax[3].imshow(cam, cmap="jet", alpha=0.5); ax[3].set_title("Grad-CAM")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("All three highlight the strokes that drive the prediction, at different resolutions.")

## 8. Surrogate Models

**Slide idea:** approximate a complex model with a simpler, *interpretable* one — e.g. a
**decision tree** as a proxy for a deep network. The surrogate is trained to mimic the black
box's predictions, then read directly.

We fit a shallow decision tree to imitate the random forest, report the fidelity (agreement),
and print the human-readable rules.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
blackbox_preds = rf.predict(Xf)                 # the model we want to explain
surrogate = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xf, blackbox_preds)
fidelity = (surrogate.predict(Xf) == blackbox_preds).mean()
print(f"Surrogate fidelity (agreement with the black box): {fidelity:.3f}\n")
print("Interpretable surrogate rules:\n")
print(export_text(surrogate, feature_names=fnames))

## 9. Attention Mechanisms

**Slide idea:** in transformers, attention weights show which input tokens are most relevant
(e.g. attention maps in BERT). We build a *minimal* self-attention layer, run a sentence
through it, and visualize the token-to-token attention matrix.

> Note: attention is a useful signal but *not a guaranteed explanation* (attention ≠
> importance in general) — we flag this at the end.

In [ ]:
sentence = "the movie was not very good".split()
V = len(sentence); d = 16
emb = nn.Embedding(V, d)
Wq, Wk, Wv = (nn.Linear(d,d) for _ in range(3))
tokens = torch.arange(V)
E = emb(tokens)
Q, Kk, Vv = Wq(E), Wk(E), Wv(E)
attn = torch.softmax(Q @ Kk.t() / d**0.5, dim=1).detach().numpy()

plt.figure(figsize=(5,4))
plt.imshow(attn, cmap="viridis")
plt.xticks(range(V), sentence, rotation=45, ha="right"); plt.yticks(range(V), sentence)
plt.colorbar(label="attention weight"); plt.title("Self-attention map (query row -> key col)")
plt.tight_layout(); plt.show()
print("Each row shows where a token attends. In real BERT these maps reveal syntactic/semantic links.")
print("Caveat: attention weights are a signal, not a faithful importance guarantee.")

---
# Conclusion

| Lecture concept | Demonstrated in |
|---|---|
| Influence functions | §1 (Hessian-based, catches mislabeled data) |
| Adversarial examples (TextFooler) | §2 (synonym swap flips sentiment) |
| Counterfactual explanations (Polyjuice) | §3 (minimal feature change) |
| Data Shapley | §4 (KNN-Shapley, negative value = bad data) |
| Prototype & criticism (MMD-critic) | §5 (typical vs. unusual digits) |
| Feature importance (SHAP/LIME/permutation) | §6 (three methods side-by-side) |
| Gradient attribution (IG/Grad-CAM/saliency) | §7 (three maps on one image) |
| Surrogate models | §8 (decision tree mimics the forest) |
| Attention mechanisms | §9 (self-attention map) |

**Key lessons, demonstrated in code:**

- **Data attribution** points *backward* to training samples (who is responsible), and is the
  practical tool for cleaning mislabeled/biased data and valuing datasets.
- **Model attribution** points *inward* to features, gradients, and attention (what drove
  this output), and is how you debug and justify a specific prediction.
- **Methods disagree, and that is informative.** Global vs. local (permutation vs. LIME/SHAP),
  gradient vs. attention — cross-checking several attributions is more trustworthy than any one.
- Attribution is central to **explaining AI models** — the bridge from a black-box prediction
  to a reason a human can inspect, contest, and act on.

---
### Ideas to extend
- Install the real `shap`, `lime`, and `captum` libraries and compare with these from-scratch versions.
- Run **TextFooler / Polyjuice** properly on a Hugging Face sentiment model.
- Use `captum`'s `IntegratedGradients` and `LayerGradCam` on a CIFAR-10 CNN.
- Add **TracIn** as a cheaper alternative to influence functions.